# Modélisation Textuelle - FastText

## Objectif
Classification des produits Rakuten en utilisant FastText pour créer des embeddings de mots et de sous-mots (n-grammes de caractères), permettant de gérer les mots hors vocabulaire.

## Approche
1. **FastText** : Entraînement d'un modèle FastText qui utilise les n-grammes de caractères
2. **Agrégation des embeddings** : Moyenne des embeddings pour représenter chaque document
3. **Classification** : Utilisation de classifieurs classiques sur les embeddings agrégés


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
warnings.filterwarnings('ignore')

# FastText
from gensim.models import FastText

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Pour sauvegarder
import joblib
import os

pd.set_option('display.max_columns', None)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

print("✅ Bibliothèques importées")


✅ Bibliothèques importées


In [2]:
# Chargement et nettoyage (identique aux autres notebooks)
X_train = pd.read_csv('X_train_update.csv')
Y_train = pd.read_csv('Y_train_CVw08PX.csv')
X_test = pd.read_csv('X_test_update.csv')

train_data = pd.merge(X_train, Y_train, left_index=True, right_index=True)

def nettoyer_texte(texte):
    if pd.isna(texte) or texte == 'nan':
        return ''
    texte = str(texte)
    texte = re.sub(r'<[^>]+>', '', texte)
    texte = texte.replace('&nbsp;', ' ')
    texte = texte.replace('&amp;', '&')
    texte = texte.replace('&lt;', '<')
    texte = texte.replace('&gt;', '>')
    texte = texte.replace('&quot;', '"')
    texte = texte.replace('&#39;', "'")
    texte = texte.replace('&eacute;', 'é')
    texte = texte.replace('&egrave;', 'è')
    texte = texte.replace('&ecirc;', 'ê')
    texte = texte.replace('&agrave;', 'à')
    texte = re.sub(r'\s+', ' ', texte)
    return texte.strip()

train_data['texte_complet'] = (
    train_data['designation'].apply(nettoyer_texte) + ' ' + 
    train_data['description'].apply(nettoyer_texte)
).str.strip()

X_test['texte_complet'] = (
    X_test['designation'].apply(nettoyer_texte) + ' ' + 
    X_test['description'].apply(nettoyer_texte)
).str.strip()

print("✅ Données chargées et nettoyées")


✅ Données chargées et nettoyées


In [3]:
# Tokenisation
def tokenize(text):
    if not text or pd.isna(text):
        return []
    tokens = text.lower().split()
    return [t for t in tokens if len(t) > 1]

sentences = [tokenize(text) for text in train_data['texte_complet']]
sentences = [s for s in sentences if len(s) > 0]

print(f"✅ {len(sentences)} phrases tokenisées")


✅ 84915 phrases tokenisées


In [4]:
# Paramètres FastText
vector_size = 100
window = 5
min_count = 2
min_n = 3  # Longueur minimale des n-grammes de caractères
max_n = 6  # Longueur maximale des n-grammes de caractères
workers = 4

print("Entraînement du modèle FastText...")
print(f"Paramètres : vector_size={vector_size}, window={window}, min_n={min_n}, max_n={max_n}")

# Entraîner FastText
ft_model = FastText(
    sentences=sentences,
    vector_size=vector_size,
    window=window,
    min_count=min_count,
    min_n=min_n,
    max_n=max_n,
    workers=workers,
    epochs=10
)

print(f"\n✅ Modèle FastText entraîné")
print(f"Vocabulaire : {len(ft_model.wv)} mots")


Entraînement du modèle FastText...
Paramètres : vector_size=100, window=5, min_n=3, max_n=6

✅ Modèle FastText entraîné
Vocabulaire : 134687 mots


In [5]:
# Fonction pour obtenir l'embedding d'un document
def document_to_vector(text, model, vector_size):
    tokens = tokenize(text)
    if len(tokens) == 0:
        return np.zeros(vector_size)
    
    vectors = []
    for token in tokens:
        # FastText peut générer un embedding même pour les mots hors vocabulaire
        vectors.append(model.wv[token])
    
    if len(vectors) == 0:
        return np.zeros(vector_size)
    
    return np.mean(vectors, axis=0)

# Créer les embeddings
print("Création des embeddings...")
X_embeddings = np.array([
    document_to_vector(text, ft_model, vector_size) 
    for text in train_data['texte_complet']
])

y = train_data['prdtypecode'].values
X_test_embeddings = np.array([
    document_to_vector(text, ft_model, vector_size) 
    for text in X_test['texte_complet']
])

print(f"✅ Embeddings créés : train {X_embeddings.shape}, test {X_test_embeddings.shape}")


Création des embeddings...
✅ Embeddings créés : train (84916, 100), test (13812, 100)


In [6]:
# Division et modélisation
X_train_split, X_val_split, y_train_split, y_val_split = train_test_split(
    X_embeddings, y, test_size=0.2, random_state=42, stratify=y
)

results = {}

# Test des classifieurs
for name, model_class in [
    ('Logistic Regression', LogisticRegression),
    ('Random Forest', RandomForestClassifier),
    ('SVM', SVC)
]:
    print(f"\n{'='*60}")
    print(name)
    print('='*60)
    
    if name == 'Logistic Regression':
        clf = model_class(max_iter=1000, class_weight='balanced', random_state=42, n_jobs=-1)
    elif name == 'Random Forest':
        clf = model_class(n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1)
    else:
        clf = model_class(kernel='linear', class_weight='balanced', random_state=42, probability=True)
    
    clf.fit(X_train_split, y_train_split)
    y_pred = clf.predict(X_val_split)
    
    results[name] = {
        'model': clf,
        'accuracy': accuracy_score(y_val_split, y_pred),
        'f1_weighted': f1_score(y_val_split, y_pred, average='weighted'),
        'f1_macro': f1_score(y_val_split, y_pred, average='macro')
    }
    print(f"Accuracy: {results[name]['accuracy']:.4f}")
    print(f"F1 (weighted): {results[name]['f1_weighted']:.4f}")

# Meilleur modèle
comparison_df = pd.DataFrame({
    'Modèle': list(results.keys()),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'F1 (weighted)': [r['f1_weighted'] for r in results.values()],
    'F1 (macro)': [r['f1_macro'] for r in results.values()]
}).sort_values('F1 (weighted)', ascending=False)

print("\n" + "="*80)
print("COMPARAISON")
print("="*80)
print(comparison_df.to_string(index=False))

# Entraîner le meilleur sur toutes les données et prédire
meilleur_nom = comparison_df.iloc[0]['Modèle']
meilleur_modele = type(results[meilleur_nom]['model'])(**results[meilleur_nom]['model'].get_params())
meilleur_modele.fit(X_embeddings, y)
y_test_pred = meilleur_modele.predict(X_test_embeddings)

# Sauvegarder
os.makedirs('models', exist_ok=True)
os.makedirs('output', exist_ok=True)

ft_model.save('models/fasttext.model')
joblib.dump(meilleur_modele, f'models/fasttext_{meilleur_nom.lower().replace(" ", "_")}.pkl')

pd.DataFrame({
    'productid': X_test['productid'],
    'prdtypecode': y_test_pred
}).to_csv('output/predictions_fasttext.csv', index=False)

print(f"\n✅ Meilleur modèle : {meilleur_nom}")
print(f"✅ Modèles et prédictions sauvegardés")



Logistic Regression
Accuracy: 0.6927
F1 (weighted): 0.6940

Random Forest
Accuracy: 0.7213
F1 (weighted): 0.7142

SVM
Accuracy: 0.7169
F1 (weighted): 0.7194

COMPARAISON
             Modèle  Accuracy  F1 (weighted)  F1 (macro)
                SVM  0.716851       0.719419    0.682524
      Random Forest  0.721326       0.714181    0.689064
Logistic Regression  0.692711       0.693964    0.646736

✅ Meilleur modèle : SVM
✅ Modèles et prédictions sauvegardés
